# 문제 3-1 : 콘텐츠분쟁해결 RAG 시스템

In [3]:
import os
from dotenv import load_dotenv
from langchain_community.document_loaders import PyPDFLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_community.vectorstores import FAISS
from langchain_upstage import ChatUpstage, UpstageEmbeddings
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser
from langchain_core.runnables import RunnablePassthrough

load_dotenv()
if not os.getenv("UPSTAGE_API_KEY"):
    raise RuntimeError(".env 파일에 UPSTAGE_API_KEY 가 없습니다.")

path = next(p for p in ["../data/콘텐츠분쟁해결_사례.pdf", "data/콘텐츠분쟁해결_사례.pdf"] if os.path.exists(p))
docs = PyPDFLoader(path).load()
print(f"문서 로드: {len(docs)}페이지")

splitter = RecursiveCharacterTextSplitter(
    chunk_size=1500,
    chunk_overlap=300,
    separators=["\n【사건개요】", "\n【쟁점사항】", "\n【처리경위】", "\n【처리결과】", "\n■", "\n\n", "\n", ".", " ", ""],
)
chunks = splitter.split_documents(docs)
print(f"문서 분할: {len(chunks)}개 청크")

emb = UpstageEmbeddings(model="solar-embedding-1-large")
idx = "faiss_content_dispute"
if os.path.exists(idx):
    vs = FAISS.load_local(idx, emb, allow_dangerous_deserialization=True)
else:
    vs = FAISS.from_documents(chunks, emb)
    vs.save_local(idx)

retriever = vs.as_retriever(search_type="similarity", search_kwargs={"k": 5})

llm = ChatUpstage(model="solar-pro3", base_url="https://api.upstage.ai/v1", temperature=0.2)

pt = ChatPromptTemplate.from_template("""
당신은 콘텐츠 분야 전문 법률 자문가입니다.
아래 분쟁조정 사례들을 바탕으로 정확하고 전문적인 법률 조언을 제공해주세요.

관련 분쟁사례:
{context}

상담 내용: {question}

답변 가이드라인:
1. 제시된 사례들을 근거로 답변하세요
2. 관련 법령이나 조항이 있다면 명시하세요
3. 비슷한 사례의 처리경위와 결과를 참고하여 설명하세요
4. 실무적 해결방안을 단계별로 제시하세요
5. 사례에 없는 내용은 "제시된 사례집에서는 확인할 수 없습니다"라고 명시하세요

전문 법률 조언:""")

def format_docs(ds):
    return "\n\n".join(f"[사례 {i}] (p.{d.metadata.get('page', 0) + 1})\n{d.page_content}" for i, d in enumerate(ds, 1))

qa_chain = (
    {"context": retriever | format_docs, "question": RunnablePassthrough()}
    | pt
    | llm
    | StrOutputParser()
)

def classify_dispute_type(query):
    game = ["게임", "아이템", "계정", "캐릭터", "레벨", "길드", "온라인게임"]
    elearning = ["강의", "온라인교육", "이러닝", "수강", "환불", "화상교육"]
    web = ["웹사이트", "무료체험", "자동결제", "구독", "사이트"]
    q = query.lower()
    if any(k in q for k in game):
        return "게임"
    elif any(k in q for k in elearning):
        return "이러닝"
    elif any(k in q for k in web):
        return "웹콘텐츠"
    return "기타"

test_questions = [
    "온라인 게임에서 시스템 오류로 아이템이 사라졌는데, 게임회사가 복구를 거부하고 있습니다. 어떻게 해결할 수 있나요?",
    "인터넷 강의를 중도 해지하려고 하는데 과도한 위약금을 요구받고 있습니다. 정당한가요?",
    "무료체험 후 자동으로 유료전환되어 요금이 청구되었습니다. 환불 가능한가요?",
    "미성년자가 부모 동의 없이 게임 아이템을 구매했습니다. 환불받을 수 있는 방법이 있나요?",
    "온라인 교육 서비스가 광고와 다르게 제공되어 계약을 해지하고 싶습니다. 가능한가요?",
]

q = input("상담 내용을 입력하세요 (Enter: 테스트 질문 5개 실행): ").strip()
qs = [q] if q else test_questions

for i, x in enumerate(qs, 1):
    print("=" * 80)
    print(f"[질문 {i}] ({classify_dispute_type(x)}) {x}")
    try:
        print(qa_chain.invoke(x))
    except Exception as e:
        print(f"오류: {type(e).__name__}: {e}")


문서 로드: 109페이지
문서 분할: 104개 청크
[질문 1] (게임) 온라인 게임에서 시스템 오류로 아이템이 사라졌는데, 게임회사가 복구를 거부하고 있습니다. 어떻게 해결할 수 있나요?
**온라인 게임 시스템 오류로 인한 아이템 소멸 – 법적·실무적 해결 방안**

---

### 1. **제시된 사례 요약 및 적용**

| 사례 | 핵심 내용 | 처리 결과 |
|------|-----------|-----------|
| **사례 1 (2006)** | 시스템 오류로 120,000원 상당의 아이템 소멸 → 계정 명의가 타인이라 복구 불가 | 계정 명의자가 직접 신청 시에만 복구 가능 |
| **사례 2 (2009)** | 업데이트 후 렉 발생, 게임머니 1,000억 소실 주장 → 시스템 오류 증거 부족, 사용 기록 정상 | 시스템 오류로 보기 어려움 → 통신·PC 문제 가능성 제시 |
| **사례 4 (2006)** | 프로그램 오류로 아이템 소멸 → 피신청인(게임사)이 복구 약속 후 이행 | 복구 완료 (아이템 및 캐릭터 복구) |
| **사례 5 (2007)** | 타 이용자가 해킹으로 분실한 아이템을 회수 → 신청인은 정상 구매 주장 | 아이템은 원소유자에게 반환, 게임머니 환급 → 신청인에게 실질적 피해 없음 |

> **공통점**: 게임사는 **약관**에 따라 계정 및 아이템 소유권을 명확히 규정하고 있으며, **시스템 오류**가 인정될 경우 복구 의무가 발생할 수 있음.  
> **차이점**: 계정 명의자 여부, 오류 발생 시점 및 증거, 피해 규모, 약관 위반 여부 등이 처리 결과에 영향을 미침.

---

### 2. **관련 법령 및 약관 조항**

#### (1) **민법**
- **제250조 (도품·유실물에 대한 특례)**  
  > “동산이 도품 또는 유실물인 때에는 피해자 또는 유실자는 도난 또는 유실한 날로부터 2년 내에 그 물건의 반환을 청구할 수 있다.”  
  → **게임 아이템은 ‘동산’으로 간주될 수 있으나, 게임사가 ‘